# Week 8 Evaluation Expansion
- Compute metrics beyond R²: MAPE and MdAPE. (https://scikit-learn.org/stable/modules/generated/sklearn.metrics.median_absolute_error.html)
    - Why is it difficult to find things on MdAPE? (https://stats.stackexchange.com/questions/596324/is-median-absolute-percentage-error-useless)
- Summarize insights (which price bands performed better)

- I'll clean up the code from previous files and re-do it here for better readability (and convenience)
- Hyperparameter tuning (https://www.geeksforgeeks.org/machine-learning/sklearn-model-hyper-parameters-tuning/)

In [ ]:
# Importing modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Hyperparameter tuning modules
from sklearn.model_selection import GridSearchCV

# Regression modules
from sklearn.linear_model import LinearRegression
from sklearn import metrics

# Tree modules
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# XGBoost
from xgboost import XGBRegressor
from scipy.stats import loguniform
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV

# Data Wrangling
Import and split data properly

In [13]:
# Importing cleaned data
root = "C:/Users/donutii/Desktop/IDX-Exchange-Internship-Data-Science-61"
data_location = f"{root}/IDX_Exchange/deliverables"

testing_set = pd.read_csv(f'{data_location}/CRMLS_202605_testing_set.csv')
training_set = pd.read_csv(f'{data_location}/CRMLS_202505_202604_training_set.csv') 

# feature engineered datasets
testing_set_fe = pd.read_csv(f'{data_location}/CRMLS_202605_testing_set_fe.csv')
training_set_fe = pd.read_csv(f'{data_location}/CRMLS_202505_202604_training_set_fe.csv') 

In [14]:
# Splitting training set into different months to test hyperparameter
months = ['2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2025-06']
training_set_months = []
for month in range(len(months)):
    training_set_months.append(training_set[training_set['CloseDate'].str.contains(months[month])])
    #(training_set_months[month].columns)
    training_set_months[month] = training_set_months[month].drop(columns='CloseDate')

# Get Target and Normal Vars from the Testing set
testing_vars = testing_set.drop(columns='ClosePrice')
testing_target = testing_set['ClosePrice']

In [15]:
# Do the same with featured engineered sets

training_set_months_fe = []
for month in range(len(months)):
    training_set_months_fe.append(training_set_fe[training_set_fe['CloseDate'].str.contains(months[month])])
    #(training_set_months[month].columns)
    training_set_months_fe[month] = training_set_months_fe[month].drop(columns='CloseDate')
    
testing_vars_fe = testing_set_fe.drop(columns='ClosePrice')
testing_target_fe = testing_set_fe['ClosePrice']

In [ ]:
""" 
    Function to test different time frames of training data
    You can include up to 13 months
"""
def choose_test_months(num_months):
    if(num_months > 13 or num_months < 1):
        num_months = 13
        
    included_months = []
    for month in range(num_months):
        included_months.append(training_set_months[12 - month])
    
    return pd.concat(included_months, axis=0)

""" 
    Function to split trianing data into its variables and target
"""
def get_training_split(training_data):
    training_vars = training_data.drop(columns='ClosePrice')
    training_target = training_data['ClosePrice']
    
    return training_vars, training_target

# Baseline Model
- From 03_baseline_model.ipynb
- A simple regression model using sklearn
- Perform hyperparameter turning via grid search

In [ ]:
def baseline_regression_model(training_vars, training_target):
    params = {
        # TODO: Find more params for grid search
    }
    
    # Create a GridSearch CV
    model_grid_searchcv = GridSearchCV(LinearRegression(), params, cv=5, scoring=['r2', 'neg_median_absolute_error', 'neg_mean_absolute_percentage_error'], n_jobs=-1)

    # Fit to the model
    model_grid_searchcv.fit(training_vars, training_target)
    
    # create cv
    columns = [f"param_{name}" for name in params.keys()]
    columns += ["r2", "MdAPE", 'MAPE']
    cv_results = pd.DataFrame(model_grid_searchcv.cv_results_)
    cv_results["r2"] = -cv_results["r2"]
    cv_results["MdAPE"] = -cv_results["neg_median_absolute_error"]
    cv_results["MAPE"] = cv_results["neg_mean_absolute_percentage_error"]
    cv_results[columns].sort_values(by="r2")
    
    # return the best model
    return model_grid_searchcv.best_estimator_

# was originally going to write in num months into the gridsearch params but i think i'll just do it in the regress normal data stage :p
    

In [ ]:
# Regress normal data

In [ ]:
# Regress feature engineered data

# Regression Decision Tree
- From 04_model_comparison.ipynb
- DecisionTree using sklearn
- DecisionTrees are also relatively cheap in processing so I will use grid search here as well

In [ ]:
def decisiontree_regression_model(training_vars, training_target):
    params = {
        "criterion" : ['squared_error', 'friedman_mse', 'poisson'],
        "depths" : [20, 30, 40, 60]
    }
    
    # Create a GridSearch CV
    model_grid_searchcv = GridSearchCV(DecisionTreeRegressor(), params, cv=5, scoring=['r2', 'neg_median_absolute_error', 'neg_mean_absolute_percentage_error'], n_jobs=-1)
    
    # Fit to training data
    model_grid_searchcv.fit(training_vars, training_target)
    
    # create cv
    columns = [f"param_{name}" for name in params.keys()]
    columns += ["r2", "MdAPE", 'MAPE']
    cv_results = pd.DataFrame(model_grid_searchcv.cv_results_)
    cv_results["r2"] = -cv_results["r2"]
    cv_results["MdAPE"] = -cv_results["neg_median_absolute_error"]
    cv_results["MAPE"] = cv_results["neg_mean_absolute_percentage_error"]
    cv_results[columns].sort_values(by="r2")
    
    # return the best model
    return model_grid_searchcv.best_estimator_

In [ ]:
# regress normal data

# Random Forest Regressor
- Also from 04_model_comparison.ipynb
- Using randomsearch because processing these takes quite a while

In [ ]:
def randomforest_regression_model(training_vars, training_target):
    params = {
        "criterion" : ['squared_error', 'friedman_mse', 'poisson'],
        "depths" : [20, 30, 40, 60],
        # TODO: get more params here
    }
    
    # Create a Random Search CV
    model_rand_searchcv = RandomizedSearchCV(RandomForestRegressor(), params, cv=5, scoring=['r2', 'neg_median_absolute_error', 'neg_mean_absolute_percentage_error'], n_jobs=-1)
    
    # Fit to training data
    model_rand_searchcv.fit(training_vars, training_target)
    
    # create cv
    columns = [f"param_{name}" for name in params.keys()]
    columns += ["r2", "MdAPE", 'MAPE']
    cv_results = pd.DataFrame(model_rand_searchcv.cv_results_)
    cv_results["r2"] = -cv_results["r2"]
    cv_results["MdAPE"] = -cv_results["neg_median_absolute_error"]
    cv_results["MAPE"] = cv_results["neg_mean_absolute_percentage_error"]
    cv_results[columns].sort_values(by="r2")
    
    # return the best model
    return model_rand_searchcv.best_estimator_

In [ ]:
# regress normal data

# XGBoost
- From 05_advanced_models.ipynb
- 

In [ ]:
def xgboost_regression_model(training_vars, training_target):
    params = {
        "max_iter": [100, 300, 500, 600, 1000],
        "max_leaf_nodes": [5, 10, 20, 50, 75, 100],
        "learning_rate": loguniform(0.01, 1),
    }
    
    # Create a GridSearch CV
    model_grid_searchcv = RandomizedSearchCV(HistGradientBoostingRegressor(), params, cv=5, scoring=['r2', 'neg_median_absolute_error', 'neg_mean_absolute_percentage_error'], n_jobs=-1)
    
    # Fit to training data
    model_grid_searchcv.fit(training_vars, training_target)
    
    # create cv
    columns = [f"param_{name}" for name in params.keys()]
    columns += ["r2", "MdAPE", 'MAPE']
    cv_results = pd.DataFrame(model_grid_searchcv.cv_results_)
    cv_results["r2"] = -cv_results["r2"]
    cv_results["MdAPE"] = -cv_results["neg_median_absolute_error"]
    cv_results["MAPE"] = cv_results["neg_mean_absolute_percentage_error"]
    cv_results[columns].sort_values(by="r2")
    
    # return the best model
    return model_grid_searchcv.best_estimator_

In [ ]:
# Regress normal data

# Consolidate evaluations
- Ideally add the best of each model to a csv and then export it

In [ ]:
# Consolidate and export evaluations here